# 01. Chunking, Embeddings & Vector Stores

**Topics covered:** Chunking Strategies · Embeddings · FAISS / Chroma

This notebook starts the **[RAG](https://github.com/S33mi/modern-ai-llm-journey/tree/main/04_rag_systems/) (Retrieval-Augmented Generation)** series.

We will:
1. Understand why we **chunk** documents
2. Apply practical **chunking strategies**
3. Create **sentence embeddings**
4. Build a **vector store** with FAISS and Chroma
5. Run simple **similarity search** over our own documents

## 1. Setup & Imports

```bash
pip install transformers sentence-transformers faiss-cpu chromadb langchain-text-splitters
```

> Works on **CPU** (Colab free tier). Use `faiss-gpu` only if you have CUDA.

In [13]:
# pip install transformers sentence-transformers faiss-cpu chromadb langchain-text-splitters
# pip install transformers sentence-transformers faiss-gpu chromadb langchain-text-splitters

In [14]:
import numpy as np
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
)
import faiss
import chromadb
from chromadb.config import Settings

print("Libraries loaded.")

Libraries loaded.


## 2. Why Chunking?

LLMs and embedding models have a **maximum context length**.  
Long documents must be split into smaller pieces (chunks) so that:

- each chunk fits inside the embedding model
- retrieval can return the *most relevant* passages, not whole files
- the final LLM prompt stays within its context window

| Strategy | Idea | Best for |
|----------|------|----------|
| Fixed size | Every *N* characters / tokens | Simple baselines |
| Recursive | Try separators in order (`\n\n`, `\n`, `.`, ` `) | General text, markdown |
| Sentence | Split on sentence boundaries | When semantics matter per sentence |
| Semantic | Group by embedding similarity | Higher quality, slower |

## 3. Sample Documents

We use a few short ML-related notes so everything runs quickly.

In [15]:
documents = [
    {
        "id": "doc1",
        "title": "Attention Mechanisms",
        "text": """
Scaled dot-product attention computes similarity between queries and keys,
scales by the square root of the key dimension, applies softmax, and uses
the resulting weights to combine values. Multi-head attention runs several
attention heads in parallel so the model can capture different types of
relationships. Causal masking prevents tokens from attending to future
positions, which is essential for autoregressive language models like GPT.
""".strip(),
    },
    {
        "id": "doc2",
        "title": "Transformers and Residuals",
        "text": """
A Transformer block typically contains multi-head self-attention followed by
a position-wise feed-forward network. Residual connections and layer
normalization stabilize training of deep stacks. Pre-LN (layer norm before
the sub-layer) is the dominant design in modern LLMs such as GPT-2, LLaMA,
and Mistral. The feed-forward network usually expands the hidden size by a
factor of four and uses GELU or SwiGLU activations.
""".strip(),
    },
    {
        "id": "doc3",
        "title": "Fine-Tuning and LoRA",
        "text": """
Full fine-tuning updates every parameter of a pre-trained model. This is
expensive in memory and storage. LoRA freezes the base weights and injects
small trainable low-rank matrices. The rank r controls capacity; alpha scales
the update. QLoRA combines 4-bit quantization of the base model with LoRA
adapters so that 7B–13B models can be fine-tuned on a single consumer GPU.
""".strip(),
    },
    {
        "id": "doc4",
        "title": "Embeddings and Similarity",
        "text": """
Sentence embeddings map text into a dense vector space where semantically
similar sentences lie close together. Cosine similarity is the standard
metric. Models such as all-MiniLM-L6-v2 and bge-small produce strong
embeddings for retrieval. Mean pooling of the last hidden states is a common
way to obtain a fixed-size sentence vector from a Transformer.
""".strip(),
    },
]

print(f"Loaded {len(documents)} documents.")
for d in documents:
    print(f"  - {d['title']} ({len(d['text'])} chars)")

Loaded 4 documents.
  - Attention Mechanisms (441 chars)
  - Transformers and Residuals (422 chars)
  - Fine-Tuning and LoRA (374 chars)
  - Embeddings and Similarity (354 chars)


## 4. Chunking Strategies

In [16]:
# --- Fixed-size character splitter ---
fixed_splitter = CharacterTextSplitter(
    chunk_size=120,
    chunk_overlap=20,
    separator=" ",
)

# --- Recursive splitter (recommended default) ---
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=30,
    separators=["\n\n", "\n", ". ", " ", ""],
)

def chunk_documents(docs, splitter):
    chunks = []
    for doc in docs:
        parts = splitter.split_text(doc["text"])
        for i, part in enumerate(parts):
            chunks.append({
                "chunk_id": f"{doc['id']}_chunk{i}",
                "doc_id": doc["id"],
                "title": doc["title"],
                "text": part.strip(),
            })
    return chunks


chunks_fixed = chunk_documents(documents, fixed_splitter)
chunks_rec = chunk_documents(documents, recursive_splitter)

print(f"Fixed-size chunks : {len(chunks_fixed)}")
print(f"Recursive chunks  : {len(chunks_rec)}")
print("\nExample recursive chunk:")
print(chunks_rec[0])

Fixed-size chunks : 17
Recursive chunks  : 12

Example recursive chunk:
{'chunk_id': 'doc1_chunk0', 'doc_id': 'doc1', 'title': 'Attention Mechanisms', 'text': 'Scaled dot-product attention computes similarity between queries and keys,\nscales by the square root of the key dimension, applies softmax, and uses'}


**Practical tips**

- `chunk_size` ≈ 200–500 tokens for many RAG setups (character size depends on language).
- `chunk_overlap` (10–20%) helps preserve context across boundaries.
- Prefer **recursive** splitting for prose and markdown.
- Keep metadata (`doc_id`, `title`, source) with every chunk for citation later.

## 5. Embeddings

We use a small, fast sentence-transformer that runs well on CPU.

In [17]:
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

chunk_texts = [c["text"] for c in chunks_rec]
embeddings = embed_model.encode(chunk_texts, show_progress_bar=True, normalize_embeddings=True)

print("Embeddings shape:", embeddings.shape)  # (num_chunks, 384)
print("First vector (8 dims):", embeddings[0][:8].round(4))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (12, 384)
First vector (8 dims): [ 0.0096 -0.0984 -0.0476  0.0161  0.01    0.0241  0.0667  0.0262]


## 6. Vector Store with FAISS

FAISS (Facebook AI Similarity Search) is a high-performance library for dense retrieval.

For normalized vectors, **inner product** = cosine similarity.

In [18]:
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)          # exact inner-product search
index.add(embeddings.astype("float32"))

print(f"FAISS index size: {index.ntotal} vectors")


def faiss_search(query: str, k: int = 3):
    q_emb = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = index.search(q_emb, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        chunk = chunks_rec[idx]
        results.append({
            "score": float(score),
            "title": chunk["title"],
            "text": chunk["text"],
            "chunk_id": chunk["chunk_id"],
        })
    return results


query = "How does multi-head attention work?"
hits = faiss_search(query, k=3)

print(f"Query: {query}\n")
for i, h in enumerate(hits, 1):
    print(f"{i}. [{h['score']:.3f}] {h['title']}")
    print(f"   {h['text'][:160]}...\n")

FAISS index size: 12 vectors
Query: How does multi-head attention work?

1. [0.802] Attention Mechanisms
   the resulting weights to combine values. Multi-head attention runs several
attention heads in parallel so the model can capture different types of...

2. [0.556] Transformers and Residuals
   A Transformer block typically contains multi-head self-attention followed by
a position-wise feed-forward network. Residual connections and layer...

3. [0.366] Attention Mechanisms
   Scaled dot-product attention computes similarity between queries and keys,
scales by the square root of the key dimension, applies softmax, and uses...



## 7. Vector Store with Chroma

Chroma is a lightweight, developer-friendly vector database. It stores embeddings **and** metadata together.

In [19]:
# In-memory client (persistent path also possible)
chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))

collection = chroma_client.get_or_create_collection(
    name="ml_notes",
    metadata={"hnsw:space": "cosine"},
)

# Upsert chunks
collection.upsert(
    ids=[c["chunk_id"] for c in chunks_rec],
    documents=[c["text"] for c in chunks_rec],
    metadatas=[{"title": c["title"], "doc_id": c["doc_id"]} for c in chunks_rec],
    embeddings=embeddings.tolist(),
)

print(f"Chroma collection count: {collection.count()}")

Chroma collection count: 12


In [20]:
def chroma_search(query: str, k: int = 3):
    q_emb = embed_model.encode([query], normalize_embeddings=True).tolist()
    res = collection.query(query_embeddings=q_emb, n_results=k)
    results = []
    for i in range(len(res["ids"][0])):
        results.append({
            "score": 1 - res["distances"][0][i],  # cosine distance → similarity
            "title": res["metadatas"][0][i]["title"],
            "text": res["documents"][0][i],
            "chunk_id": res["ids"][0][i],
        })
    return results


query2 = "What is LoRA and why is it useful?"
hits2 = chroma_search(query2, k=3)

print(f"Query: {query2}\n")
for i, h in enumerate(hits2, 1):
    print(f"{i}. [{h['score']:.3f}] {h['title']}")
    print(f"   {h['text'][:160]}...\n")

Query: What is LoRA and why is it useful?

1. [0.525] Fine-Tuning and LoRA
   Full fine-tuning updates every parameter of a pre-trained model. This is
expensive in memory and storage. LoRA freezes the base weights and injects...

2. [0.374] Fine-Tuning and LoRA
   the update. QLoRA combines 4-bit quantization of the base model with LoRA
adapters so that 7B–13B models can be fine-tuned on a single consumer GPU....

3. [0.209] Transformers and Residuals
   and Mistral. The feed-forward network usually expands the hidden size by a
factor of four and uses GELU or SwiGLU activations....



## 8. FAISS vs Chroma – When to Use Which

| Feature | FAISS | Chroma |
|---------|-------|--------|
| Speed at scale | Excellent | Good |
| Metadata filtering | Manual | Built-in |
| Persistence | DIY (save index) | Easy (path or server) |
| Ease of use | Lower-level | Higher-level API |
| Typical use | Research, large corpora | Apps, prototypes, RAG pipelines |

Many production stacks use **Chroma / LanceDB / Qdrant / Pinecone** for the app layer and FAISS-style indexes under the hood.

## 9. End-to-End Mini Retrieval Function

In [21]:
def retrieve(query: str, k: int = 2, backend: str = "chroma"):
    """Return top-k chunks for a query."""
    if backend == "faiss":
        return faiss_search(query, k=k)
    return chroma_search(query, k=k)


for q in [
    "Explain residual connections in transformers",
    "How are sentence embeddings created?",
]:
    print(f"\n=== {q} ===")
    for h in retrieve(q, k=2):
        print(f"  [{h['score']:.3f}] {h['title']}: {h['text'][:100]}...")


=== Explain residual connections in transformers ===
  [0.531] Transformers and Residuals: A Transformer block typically contains multi-head self-attention followed by
a position-wise feed-fo...
  [0.244] Embeddings and Similarity: way to obtain a fixed-size sentence vector from a Transformer....

=== How are sentence embeddings created? ===
  [0.612] Embeddings and Similarity: Sentence embeddings map text into a dense vector space where semantically
similar sentences lie clos...
  [0.397] Embeddings and Similarity: way to obtain a fixed-size sentence vector from a Transformer....


## 10. Summary

| Step | Tool / idea |
|------|-------------|
| **Chunk** | `RecursiveCharacterTextSplitter` (size + overlap) |
| **Embed** | `SentenceTransformer` (e.g. `all-MiniLM-L6-v2`) |
| **Index** | FAISS (`IndexFlatIP`) or Chroma collection |
| **Search** | Encode query → nearest neighbours by cosine / IP |
| **Metadata** | Keep `doc_id`, title, source with every chunk |

### Canonical pattern

```python
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
chunks = splitter.split_text(long_document)

model = SentenceTransformer("all-MiniLM-L6-v2")
emb = model.encode(chunks, normalize_embeddings=True)

index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb.astype("float32"))

q = model.encode(["my question"], normalize_embeddings=True)
scores, idxs = index.search(q.astype("float32"), k=5)
```

---

**Next notebook:** [`02_basic_rag_pipeline.ipynb`](https://github.com/S33mi/modern-ai-llm-journey/tree/main/04_rag_systems/02_basic_rag_pipeline.ipynb)
Retrieval + Generation · Prompt Engineering for RAG

---

**For contribution and insihght:** [**S33mi**](https://github.com/S33mi)

Open to Data Analytics and ML/AI related opportunities